In [ ]:
#| default_exp machine_learning.ml_data

> **AI-generated — pending human review.** Created by Claude Code on 2026-10-21. Review checklist: `_nbdev_development/AI_REVIEW_CHECKLIST.md` (workspace root). Remove this cell only after review passes.

Training datasets are produced inside the vault (front-end utility notebooks writing CSVs/Arrow dirs under `_ml_data/`) and consumed by trainer notebooks in `../trouver_personal_playground/`. The columns each side expects are an *implicit* contract, versioned only by a date embedded in a directory name — so producer drift surfaces as a crash deep inside a training run, or worse, silently trains on the wrong column.

This module makes the contract explicit: a `DatasetSchema` pins each dataset's columns and version, `dataset_dir` fixes the `_ml_data/<name>_v<version>` path convention (versioned by *schema*, not date), and `save_rows`/`load_rows` give both sides one stdlib-only interchange format. Producers and trainers import the same schema object; drift becomes an error at the load boundary instead.

In [ ]:
#| export
import csv
import json
import random
from dataclasses import dataclass
from os import PathLike
from pathlib import Path
from typing import Callable

In [ ]:
#| export
ML_DATA_DIR_NAME = '_ml_data'

PROVENANCE_COLUMN = 'Note name'
"""Column every dataset row carries: the vault note the row was derived from.
`machine_learning.train_split.split_list_consistently` splits on it, so one
note is in train or validation once, for *every* task."""

AUGMENTATION_COLUMN = 'augmentation'
"""Column labeling how a row was produced. `DatasetSchema.validate_rows`
always allows it — it is a purity label, not schema drift."""

PURE_LABEL = 'pure'
"""Value of `AUGMENTATION_COLUMN` for rows extracted from the vault with no
synthetic modification. A missing label counts as pure (pre-augmentation
datasets). Frozen eval sets and augmentation inputs require it."""

In [ ]:
#| export
@dataclass(frozen=True)
class DatasetSchema:
    """
    Pin the columns and version of one ML training dataset.

    **Parameters**
    - `name` (*str*) — Dataset name, e.g. `'note_type'`; also the path component.
    - `version` (*int*) — Schema version; bump whenever columns change so old and new generations coexist on disk.
    - `required_columns` (*tuple*) — Columns every row must have (include `PROVENANCE_COLUMN`).
    - `optional_columns` (*tuple*) — Columns a row may carry.
    """
    name: str
    version: int
    required_columns: tuple
    optional_columns: tuple = ()

    def validate_rows(self, rows: list[dict], allow_extra: bool = False) -> None:
        """
        Raise `ValueError` unless every row matches this schema.

        An empty row list is valid: a fresh dataset has no columns to check.
        `AUGMENTATION_COLUMN` is always allowed — the purity label is part of
        the storage format, not a column each producer must declare.

        **Parameters**
        - `rows` (*list[dict]*) — Rows as produced by a producer (or read back by `load_rows`).
        - `allow_extra` (*bool*) — Permit columns beyond those pinned here.
        """
        if not rows:
            return
        present = set().union(*(row.keys() for row in rows))
        missing = sorted(set(self.required_columns) - present)
        allowed = set(self.required_columns) | set(self.optional_columns) | {AUGMENTATION_COLUMN}
        extra = [] if allow_extra else sorted(present - allowed)
        if missing or extra:
            raise ValueError(f'Dataset {self.name!r} v{self.version} mismatch: '
                             f'missing={missing}, unexpected={extra}.')

In [ ]:
#| export
def dataset_dir(root: PathLike, schema: DatasetSchema) -> Path:
    """
    Return `<root>/_ml_data/<name>_v<version>` for `schema`.

    Versioned by schema instead of the old `_<date>` suffix, so two
    generations of a dataset coexist and the right one is named by the
    code that loads it, not by a dated path string.

    **Parameters**
    - `root` (*PathLike*) — Usually the vault directory.
    - `schema` (*DatasetSchema*) — The dataset whose directory is wanted.
    """
    return Path(root) / ML_DATA_DIR_NAME / f'{schema.name}_v{schema.version}'

In [ ]:
#| export
def save_rows(rows: list[dict], path: PathLike) -> None:
    """
    Write `rows` as JSONL (one JSON object per line), creating parents.

    JSONL is the schema-checked interchange format between producers and
    trainers: diff-able, dependency-free, and convertible to a HF
    `Dataset` with `Dataset.from_list` at the consumer's choice.

    **Parameters**
    - `rows` (*list[dict]*) — Rows to write.
    - `path` (*PathLike*) — Destination `.jsonl` file.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8', newline='') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

In [ ]:
#| export
def load_rows(path: PathLike, schema: DatasetSchema = None, require_pure: bool = False) -> list[dict]:
    """
    Read JSONL written by `save_rows`, optionally validating against `schema`.

    Passing the same `DatasetSchema` the producer used turns any column
    drift into an immediate `ValueError` at the load boundary, instead of
    a training run on the wrong data. With `require_pure=True`, rows
    labeled with any augmentation other than `PURE_LABEL` are refused —
    frozen eval sets and `augment_dataset` inputs pass this flag (R5).

    **Parameters**
    - `path` (*PathLike*) — A `.jsonl` file written by `save_rows`.
    - `schema` (*DatasetSchema*) — Contract to validate the rows against.
    - `require_pure` (*bool*) — Refuse augmented rows; a missing label counts as pure.
    """
    with open(path, 'r', encoding='utf-8') as f:
        rows = [json.loads(line) for line in f if line.strip()]
    if schema is not None:
        schema.validate_rows(rows)
    if require_pure:
        impure = sorted({r.get(AUGMENTATION_COLUMN, PURE_LABEL) for r in rows} - {PURE_LABEL})
        if impure:
            raise ValueError(f'{path}: require_pure but found augmented rows {impure}.')
    return rows

In [ ]:
#| export
AUGMENTATION_RECIPES: dict[str, Callable] = {}


def register_augmentation_recipe(name: str, version: int = 1):
    """
    Decorator registering a row-level augmentation recipe as `<name>_v<version>`.

    A recipe is `fn(row: dict, rng: random.Random) -> dict | None`; returning
    `None` drops the row. Recipes must use the given `rng` and never the
    global `random` module, so an augmented dataset is exactly regenerable
    from (pure dataset, recipe, seed).

    **Parameters**
    - `name` (*str*) — Recipe name, e.g. `'latex_typo'`.
    - `version` (*int*) — Bump when the algorithm changes; old ids keep resolving to old code paths only if kept registered.
    """
    def decorator(fn: Callable) -> Callable:
        AUGMENTATION_RECIPES[f'{name}_v{version}'] = fn
        return fn
    return decorator


def get_augmentation_recipe(recipe_id: str) -> Callable:
    """
    Return the recipe registered as `recipe_id`, failing with the list of known recipes.

    **Parameters**
    - `recipe_id` (*str*) — Key of `AUGMENTATION_RECIPES`, e.g. `'latex_typo_v1'`.
    """
    if recipe_id not in AUGMENTATION_RECIPES:
        raise KeyError(f'Unknown augmentation recipe {recipe_id!r}; known: {sorted(AUGMENTATION_RECIPES)}.')
    return AUGMENTATION_RECIPES[recipe_id]

In [ ]:
#| export
def augmented_dataset_dir(root: PathLike, schema: DatasetSchema, recipe_id: str) -> Path:
    """
    Return `<root>/_ml_data/<name>_aug-<recipe_id>_v<version>` for `schema`.

    The directory name itself says *augmented* — an augmented dataset can
    never be mistaken for the pure one at the path level (R5).

    **Parameters**
    - `root` (*PathLike*) — Usually the vault directory.
    - `schema` (*DatasetSchema*) — Schema of the underlying pure dataset.
    - `recipe_id` (*str*) — Key of `AUGMENTATION_RECIPES`, e.g. `'latex_typo_v1'`.
    """
    return Path(root) / ML_DATA_DIR_NAME / f'{schema.name}_aug-{recipe_id}_v{schema.version}'


def augment_dataset(
        root: PathLike, schema: DatasetSchema, recipe_id: str,
        seed: int = 0, validate: Callable | None = None) -> list[dict]:
    """
    Apply a registered recipe to a pure dataset, writing an augmented sibling dir.

    The input must be pure (`require_pure`) — augmentation never runs on
    augmented data. Each output row is stamped `augmentation=recipe_id` and
    inherits its source's `PROVENANCE_COLUMN`, so train/valid splits always
    place a paraphrase on the same side as its original (no leakage). The
    result is regenerable from (pure dataset, recipe, seed) alone.

    **Parameters**
    - `root` (*PathLike*) — Vault root holding `_ml_data/`.
    - `schema` (*DatasetSchema*) — Schema of the pure dataset.
    - `recipe_id` (*str*) — Key of `AUGMENTATION_RECIPES`.
    - `seed` (*int*) — Seed for the recipe RNG (regenerability).
    - `validate` (*Callable*) — Optional `fn(original, augmented) -> bool` guard; rows it rejects are dropped (e.g. a LaTeX-preservation check).
    """
    recipe = get_augmentation_recipe(recipe_id)
    rows = load_rows(dataset_dir(root, schema) / f'{schema.name}.jsonl', schema=schema, require_pure=True)
    rng = random.Random(seed)
    out = []
    for row in rows:
        new_row = recipe(dict(row), rng)
        if new_row is None or (validate is not None and not validate(row, new_row)):
            continue
        new_row[PROVENANCE_COLUMN] = row[PROVENANCE_COLUMN]
        new_row[AUGMENTATION_COLUMN] = recipe_id
        out.append(new_row)
    save_rows(out, augmented_dataset_dir(root, schema, recipe_id) / f'{schema.name}.jsonl')
    return out

In [ ]:
#| export
def load_training_rows(*paths: PathLike, schema: DatasetSchema = None) -> list[dict]:
    """
    Concatenate dataset files (pure + chosen augmented ones) into one training set.

    Because pure and augmented rows live in separate files (R5), each
    training input states explicitly what it contains; concatenation order
    is argument order, so runs stay deterministic.

    **Parameters**
    - `paths` (*PathLike*) — `.jsonl` files from `dataset_dir`/`augmented_dataset_dir`, in the order they should be concatenated.
    - `schema` (*DatasetSchema*) — Contract every file's rows are validated against.
    """
    rows = []
    for path in paths:
        rows.extend(load_rows(path, schema=schema))
    return rows

In [ ]:
#| export
def save_csv(rows: list[dict], path: PathLike) -> None:
    """
    Write `rows` as CSV — the human-facing view of a dataset (R6).

    Values must be flat scalars: a row with nested values (token/label
    lists) raises `TypeError` instead of being stringified into silently
    corrupt data. CSV reads back every value as a *string*, so JSONL stays
    the canonical format; use this for spreadsheets and eyeballing, not for
    round-tripping types.

    **Parameters**
    - `rows` (*list[dict]*) — Rows to write; the first row fixes the column order.
    - `path` (*PathLike*) — Destination `.csv` file.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text('', encoding='utf-8', newline='')
        return
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        for row in rows:
            if any(isinstance(v, (list, dict)) for v in row.values()):
                raise TypeError('save_csv supports flat values only; use JSONL for nested rows.')
            writer.writerow(row)


def load_csv(path: PathLike, schema: DatasetSchema = None) -> list[dict]:
    """
    Read a `save_csv` file back (all values as strings), validating against `schema` if given.

    **Parameters**
    - `path` (*PathLike*) — A `.csv` file written by `save_csv`.
    - `schema` (*DatasetSchema*) — Contract to validate the rows against.
    """
    with open(path, 'r', encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))
    if schema is not None:
        schema.validate_rows(rows)
    return rows

In [ ]:
#| export
def as_hf_dataset(source, schema: DatasetSchema = None):
    """
    Return an HF `datasets.Dataset` for `source` (a list of rows or a JSONL path).

    `datasets` is imported lazily so merely importing this module — e.g. in
    the GUI to check a schema — never pulls training dependencies (R6).

    **Parameters**
    - `source` (*list[dict] | PathLike*) — Rows in memory, or a `.jsonl` file written by `save_rows`.
    - `schema` (*DatasetSchema*) — Contract to validate file rows against.
    """
    from datasets import Dataset
    rows = source if isinstance(source, list) else load_rows(source, schema=schema)
    return Dataset.from_list(rows)


def from_hf_dataset(ds, path: PathLike, schema: DatasetSchema = None) -> None:
    """
    Write an HF `Dataset` back to canonical JSONL via `save_rows`.

    **Parameters**
    - `ds` — A Hugging Face `Dataset` (anything iterable of dicts).
    - `path` (*PathLike*) — Destination `.jsonl` file.
    - `schema` (*DatasetSchema*) — Contract to validate the rows against before writing.
    """
    rows = [dict(row) for row in ds]
    if schema is not None:
        schema.validate_rows(rows)
    save_rows(rows, path)

In [ ]:
import tempfile
from pathlib import PurePosixPath
from unittest import mock

from fastcore.test import *

In [ ]:
schema = DatasetSchema('demo', 2, ('Note name', 'text'), ('label',))

# Good rows (optional column present or absent) validate silently.
schema.validate_rows([{'Note name': 'a', 'text': 't'},
                      {'Note name': 'b', 'text': 't', 'label': 1}])

# Missing required and unexpected columns are both reported.
bad = [{'Note name': 'a', 'text': 't', 'typo_col': 0}]
with ExceptionExpected(regex=r"missing=\['text'\]"):
    schema.validate_rows([{'Note name': 'a'}])
with ExceptionExpected(regex='typo_col'):
    schema.validate_rows(bad)
# allow_extra tolerates the unexpected column; missing is still fatal.
schema.validate_rows(bad, allow_extra=True)

# Empty row lists are vacuously valid (a fresh dataset dir).
schema.validate_rows([])

In [ ]:
#| hide
d = dataset_dir('/vault', DatasetSchema('note_type', 1, ('Note name',)))
test_eq(PurePosixPath(str(d).replace('\\', '/')), PurePosixPath('/vault/_ml_data/note_type_v1'))

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    schema = DatasetSchema('demo', 1, ('Note name', 'text'))
    rows = [{'Note name': 'my_note', 'text': '$\\dim$ is an operator.'},
            {'Note name': 'other', 'text': 'plain'}]
    path = Path(tmp) / 'nested' / 'demo.jsonl'
    save_rows(rows, path)
    test_eq(load_rows(path, schema=schema), rows)

    # A drifted producer (renamed column) fails at the load boundary.
    drifted = [{'Note name': 'my_note', 'context': '$\\dim$'}]
    save_rows(drifted, path)
    with ExceptionExpected(regex='missing=.*text'):
        load_rows(path, schema=schema)

In [ ]:
@register_augmentation_recipe('upper', version=1)
def _upper_recipe(row, rng):
    return {**row, 'text': row['text'].upper()}

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    schema = DatasetSchema('demo', 1, ('Note name', 'text'))
    pure = [{'Note name': 'a', 'text': 'x $\\alpha$'}, {'Note name': 'b', 'text': 'y'}]
    save_rows(pure, dataset_dir(root, schema) / 'demo.jsonl')

    out = augment_dataset(root, schema, 'upper_v1')
    test_eq([r['text'] for r in out], ['X $\\ALPHA$', 'Y'])
    test_eq([r[AUGMENTATION_COLUMN] for r in out], ['upper_v1'] * 2)
    test_eq([r[PROVENANCE_COLUMN] for r in out], ['a', 'b'])  # Provenance inherited.

    aug_file = augmented_dataset_dir(root, schema, 'upper_v1') / 'demo.jsonl'
    assert '_aug-upper_v1_' in str(aug_file) and aug_file.exists()
    test_eq(load_rows(aug_file, schema=schema), out)  # Label passes validation.

    # Strict separation: augmented files are refused by require_pure; the
    # (unlabeled) pure file passes it.
    with ExceptionExpected(regex='require_pure'):
        load_rows(aug_file, require_pure=True)
    test_eq(len(load_rows(dataset_dir(root, schema) / 'demo.jsonl', require_pure=True)), 2)

    # Concatenation for training keeps explicit order.
    mixed = load_training_rows(dataset_dir(root, schema) / 'demo.jsonl', aug_file)
    test_eq([r.get(AUGMENTATION_COLUMN, PURE_LABEL) for r in mixed], ['pure', 'pure', 'upper_v1', 'upper_v1'])

    # Unknown recipes fail loudly, listing the known ones.
    with ExceptionExpected(regex='upper_v1'):
        get_augmentation_recipe('nope_v9')

In [ ]:
from trouver.machine_learning.train_split import get_stable_hash_modulo

@register_augmentation_recipe('coinflip', version=1)
def _coin(row, rng):
    return {**row, 'text': f'{rng.random():.6f} ' + row['text']}

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    schema = DatasetSchema('demo', 1, ('Note name', 'text'))
    pure = [{'Note name': f'n{i}', 'text': 'body'} for i in range(20)]
    save_rows(pure, dataset_dir(root, schema) / 'demo.jsonl')

    # Regenerable: same (pure, recipe, seed) gives the identical dataset; a
    # different seed gives a different one.
    out1 = augment_dataset(root, schema, 'coinflip_v1', seed=7)
    test_eq(out1, augment_dataset(root, schema, 'coinflip_v1', seed=7))
    test_ne([r['text'] for r in out1],
            [r['text'] for r in augment_dataset(root, schema, 'coinflip_v1', seed=8)])

    # Split safety: augmented rows inherit the note name, so they always land
    # on the same split side as their original (no train/valid leakage).
    for orig, aug in zip(pure, out1):
        test_eq(get_stable_hash_modulo(orig[PROVENANCE_COLUMN]),
                get_stable_hash_modulo(aug[PROVENANCE_COLUMN]))

    # The validate hook drops rows; rejecting all still writes an empty dataset.
    test_eq(augment_dataset(root, schema, 'coinflip_v1', seed=7,
                            validate=lambda o, n: False), [])

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    schema = DatasetSchema('demo', 1, ('Note name', 'text'))
    rows = [{'Note name': 'a', 'text': 'hello, "world"'}, {'Note name': 'b', 'text': 'plain'}]
    path = Path(tmp) / 'nested' / 'demo.csv'
    save_csv(rows, path)
    test_eq(load_csv(path, schema=schema), rows)  # String values round-trip exactly.

    # Nested values are refused loudly instead of being str()-ed into the file.
    ExceptionExpected(lambda: save_csv([{'Note name': 'a', 'tokens': ['x']}], path))

    # Column drift is checked on CSV load too.
    with ExceptionExpected(regex='missing=.*label'):
        load_csv(path, schema=DatasetSchema('demo', 1, ('Note name', 'text', 'label')))

    # An empty dataset writes an empty file and reads back as [].
    save_csv([], path)
    test_eq(load_csv(path), [])

In [ ]:
try:
    import datasets
except ImportError:
    datasets = None

# Skipped silently when `datasets` is not installed in this environment;
# the converters only lazy-import it, so the rest of the module always tests.
if datasets is not None:
    with tempfile.TemporaryDirectory() as tmp:
        schema = DatasetSchema('demo', 1, ('Note name', 'text'))
        rows = [{'Note name': 'a', 'text': 'x'}, {'Note name': 'b', 'text': 'y'}]

        ds = as_hf_dataset(rows)
        test_eq(len(ds), 2)
        path = Path(tmp) / 'demo.jsonl'
        from_hf_dataset(ds, path, schema=schema)
        test_eq(load_rows(path, schema=schema), rows)

        # From a JSONL path directly, with schema validation.
        test_eq(len(as_hf_dataset(path, schema=schema)), 2)